In [1]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

loader=PyPDFLoader("/Users/ysyseom/langchain-llm-projects/02-pdf-rag-chatbot/attention_is_all_you_need.pdf")
pages = loader.load()

/var/folders/3t/w90p5wt14m17gr1007xzm2p40000gn/T/ipykernel_12912/3033526776.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/ysyseom/Library/Caches/pypoetry/virtualenvs/pdf-bot-_gPoQf9i-py3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    # Set a really small chunk size, just to show.
    chunk_size=1000,
    chunk_overlap=200,
)

In [3]:
splits = text_splitter.split_documents(pages)

len(splits)


52

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

db = Chroma.from_documents(splits, embeddings_model)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9780.92it/s]


In [5]:
#유사도 검색
similarity_retriever = db.as_retriever(
    search_type ='similarity',
    search_kwargs={'k':3}
)


In [6]:
# MMR 검색
mmr_retriever = db.as_retriever(
    search_type='mmr',
    search_kwargs = {'k':3, 'fetch_k':10})

In [7]:
#prompt
from langchain_core.prompts import ChatPromptTemplate

template="""Answer the question based only on the following context:
            <context>
            {context}
            </context>
            Question: {input}
            """
prompt = ChatPromptTemplate.from_template(template)

In [8]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n            <context>\n            {context}\n            </context>\n            Question: {input}\n            '), additional_kwargs={})])

In [11]:
from langchain_groq import ChatGroq
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, api_key=GROQ_API_KEY)

document_chain = create_stuff_documents_chain(model, prompt)
similarity_retriever_chain = create_retrieval_chain(similarity_retriever, document_chain)

similarity_response = similarity_retriever_chain.invoke({"input":"What is the attention mechanism in transformers?"})

In [12]:
similarity_response

{'input': 'What is the attention mechanism in transformers?',
 'context': [Document(metadata={'page_label': '5', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'keywords': '', 'title': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'subject': '', 'trapped': '/False', 'page': 4, 'creator': 'LaTeX with hyperref', 'producer': 'pdfTeX-1.40.25', 'total_pages': 15, 'source': '/Users/ysyseom/langchain-llm-projects/02-pdf-rag-chatbot/attention_is_all_you_need.pdf'}, page_content='The Transformer uses multi-head attention in three different ways:\n• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,\nand the memory keys and values come from the output of the encoder. This allows every\nposition in the decoder to attend over all positions in the input sequence. This mimics the\ntypical encoder-decoder attention mechanisms in sequence-to-seque

In [13]:
mmr_retrieval_chain = create_retrieval_chain(mmr_retriever, document_chain)

mmr_response = mmr_retrieval_chain.invoke({"input": "what is the attention mechanism in transformers?"})

mmr_response

{'input': 'what is the attention mechanism in transformers?',
 'context': [Document(metadata={'creator': 'LaTeX with hyperref', 'author': '', 'creationdate': '2023-08-03T00:07:29+00:00', 'subject': '', 'total_pages': 15, 'moddate': '2023-08-03T00:07:29+00:00', 'page_label': '5', 'producer': 'pdfTeX-1.40.25', 'keywords': '', 'trapped': '/False', 'title': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': '/Users/ysyseom/langchain-llm-projects/02-pdf-rag-chatbot/attention_is_all_you_need.pdf', 'page': 4}, page_content='The Transformer uses multi-head attention in three different ways:\n• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,\nand the memory keys and values come from the output of the encoder. This allows every\nposition in the decoder to attend over all positions in the input sequence. This mimics the\ntypical encoder-decoder attention mechanisms in sequence-to-seque